# Pandas 入门：先从一张现实但干净的表开始

这一版先避开脏数据清洗。第一阶段只使用年度财务表，目标是做出一份 2020 年公司表现排名。

财务表中有 `证券代码`，但第一阶段暂时不处理它：编号列不参与计算，先从更直观的列开始练习 pandas 的基本操作。第二阶段再正式处理证券代码，并把财务表和公司信息表合并。

## 阶段 1：用一张财务表做出 2020 年公司表现排名

先读入 `finance_teaching_clean.xlsx`。这张表每行是一家公司某一年的财务数据。

In [1]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", 20)

finance_raw = pd.read_excel("data/finance_teaching_clean.xlsx")
finance_raw.head()

,证券代码,证券简称,年份,总资产_亿元,营业收入_亿元,净利润_亿元,资产负债率
0,1,平安银行,2018,34185.92,1062.12,248.18,0.9298
1,1,平安银行,2019,39390.70,1268.14,281.95,0.9205
2,1,平安银行,2020,44685.14,1432.42,289.28,0.9185
3,2,万科A,2018,15285.79,2976.79,492.72,0.8459
4,2,万科A,2019,17299.29,3678.94,551.32,0.8436


`证券代码` 是编号，不是财务指标。第一阶段先把它放到一边，集中练习选行、选列、计算和排序。

In [2]:
finance = finance_raw.drop(columns=["证券代码"])
finance.head()

,证券简称,年份,总资产_亿元,营业收入_亿元,净利润_亿元,资产负债率
0,平安银行,2018,34185.92,1062.12,248.18,0.9298
1,平安银行,2019,39390.70,1268.14,281.95,0.9205
2,平安银行,2020,44685.14,1432.42,289.28,0.9185
3,万科A,2018,15285.79,2976.79,492.72,0.8459
4,万科A,2019,17299.29,3678.94,551.32,0.8436


先看这张表的基本结构。

In [3]:
print("行列数：", finance.shape)
print("列名：", finance.columns.tolist())

finance.info()

行列数： (84, 6)
列名： ['证券简称', '年份', '总资产_亿元', '营业收入_亿元', '净利润_亿元', '资产负债率']
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 84 entries, 0 to 83
Data columns (total 6 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   证券简称     84 non-null     object 
 1   年份       84 non-null     int64  
 2   总资产_亿元   84 non-null     float64
 3   营业收入_亿元  84 non-null     float64
 4   净利润_亿元   84 non-null     float64
 5   资产负债率    84 non-null     float64
dtypes: float64(4), int64(1), object(1)
memory usage: 4.1+ KB


选出我们关心的几列。

In [4]:
finance[["证券简称", "年份", "营业收入_亿元", "净利润_亿元"]].head()

,证券简称,年份,营业收入_亿元,净利润_亿元
0,平安银行,2018,1062.12,248.18
1,平安银行,2019,1268.14,281.95
2,平安银行,2020,1432.42,289.28
3,万科A,2018,2976.79,492.72
4,万科A,2019,3678.94,551.32


筛选 2020 年的数据。

In [5]:
finance_2020 = finance[finance["年份"] == 2020]
finance_2020.head()

,证券简称,年份,总资产_亿元,营业收入_亿元,净利润_亿元,资产负债率
2,平安银行,2020,44685.14,1432.42,289.28,0.9185
5,万科A,2020,18691.77,4191.12,592.98,0.8128
8,国华网安,2020,15.64,2.81,0.62,0.0683
11,深振业A,2020,154.35,29.35,9.03,0.4922
14,*ST 全新,2020,3.64,0.45,-1.23,0.8139


在原有列的基础上计算新指标。

In [6]:
finance["净利率"] = finance["净利润_亿元"] / finance["营业收入_亿元"]
finance["资产收益率"] = finance["净利润_亿元"] / finance["总资产_亿元"]
finance["是否盈利"] = finance["净利润_亿元"] > 0

finance[["证券简称", "年份", "净利率", "资产收益率", "是否盈利"]].head()

,证券简称,年份,净利率,资产收益率,是否盈利
0,平安银行,2018,0.233665,0.007260,True
1,平安银行,2019,0.222333,0.007158,True
2,平安银行,2020,0.201952,0.006474,True
3,万科A,2018,0.165521,0.032234,True
4,万科A,2019,0.149858,0.031870,True


重新筛选 2020 年，并按营业收入排序。

In [7]:
finance_2020_rank = (
    finance[finance["年份"] == 2020]
    .sort_values("营业收入_亿元", ascending=False)
)

finance_2020_rank[[
    "证券简称", "营业收入_亿元", "净利润_亿元", "净利率", "资产收益率", "资产负债率"
]].head(10)

,证券简称,营业收入_亿元,净利润_亿元,净利率,资产收益率,资产负债率
5,万科A,4191.12,592.98,0.141485,0.031724,0.8128
47,美的集团,2857.10,275.07,0.096276,0.076327,0.6553
50,潍柴动力,1974.91,112.75,0.057091,0.041644,0.7029
77,浦发银行,1746.87,589.93,0.337707,0.007420,0.9188
83,民生银行,1667.77,351.02,0.210473,0.005050,0.9221
2,平安银行,1432.42,289.28,0.201952,0.006474,0.9185
80,华夏银行,927.17,215.68,0.232622,0.006344,0.9169
20,深康佳A,503.52,5.40,0.010724,0.010827,0.7851
65,江铃汽车,330.96,5.51,0.016649,0.019549,0.6102
62,云南白药,327.43,55.11,0.168311,0.099803,0.3056


做几个简单统计，了解 2020 年样本公司的整体情况。

In [8]:
finance_2020[["总资产_亿元", "营业收入_亿元", "净利润_亿元", "资产负债率"]].describe()

,总资产_亿元,营业收入_亿元,净利润_亿元,资产负债率
count,28.000000,28.000000,28.000000,28.000000
mean,9138.399286,614.462143,91.695357,0.595207
std,21375.065189,1025.949436,173.201886,0.235975
min,3.640000,0.450000,-1.230000,0.068300
25%,102.247500,33.870000,0.610000,0.455550
50%,154.625000,109.975000,6.410000,0.642550
75%,1091.017500,609.432500,69.520000,0.723450
max,79502.180000,4191.120000,592.980000,0.922100


In [9]:
print("公司数量：", finance_2020["证券简称"].nunique())
print("营业收入平均值：", finance_2020["营业收入_亿元"].mean())
print("营业收入最大值：", finance_2020["营业收入_亿元"].max())

公司数量： 28
营业收入平均值： 614.4621428571428
营业收入最大值： 4191.12


阶段 1 小结：这一阶段只讲基础功能，包括读取、临时放下暂不处理的列、查看、选列、筛行、列运算、排序和简单统计。

## 阶段 2：处理证券代码，并加入公司信息

现在要把财务排名和公司信息表合并。合并时需要可靠的连接键，所以这一阶段必须认真处理 `证券代码`：把它作为字符串，并补齐到 6 位。

In [10]:
def read_code(x):
    return str(x).strip().zfill(6)

finance_with_code = pd.read_excel(
    "data/finance_teaching_clean.xlsx",
    converters={"证券代码": read_code},
)

company = pd.read_excel(
    "data/company_profile_teaching_clean.xlsx",
    converters={"证券代码": read_code},
)

finance_with_code.head()

,证券代码,证券简称,年份,总资产_亿元,营业收入_亿元,净利润_亿元,资产负债率
0,000001,平安银行,2018,34185.92,1062.12,248.18,0.9298
1,000001,平安银行,2019,39390.70,1268.14,281.95,0.9205
2,000001,平安银行,2020,44685.14,1432.42,289.28,0.9185
3,000002,万科A,2018,15285.79,2976.79,492.72,0.8459
4,000002,万科A,2019,17299.29,3678.94,551.32,0.8436


先在带代码的财务表中重新生成阶段 1 用过的指标。

In [11]:
finance_with_code["净利率"] = finance_with_code["净利润_亿元"] / finance_with_code["营业收入_亿元"]
finance_with_code["资产收益率"] = finance_with_code["净利润_亿元"] / finance_with_code["总资产_亿元"]
finance_with_code["是否盈利"] = finance_with_code["净利润_亿元"] > 0

finance_2020_rank_with_code = (
    finance_with_code[finance_with_code["年份"] == 2020]
    .sort_values("营业收入_亿元", ascending=False)
)

finance_2020_rank_with_code.head()

,证券代码,证券简称,年份,总资产_亿元,营业收入_亿元,净利润_亿元,资产负债率,净利率,资产收益率,是否盈利
5,000002,万科A,2020,18691.77,4191.12,592.98,0.8128,0.141485,0.031724,True
47,000333,美的集团,2020,3603.83,2857.10,275.07,0.6553,0.096276,0.076327,True
50,000338,潍柴动力,2020,2707.50,1974.91,112.75,0.7029,0.057091,0.041644,True
77,600000,浦发银行,2020,79502.18,1746.87,589.93,0.9188,0.337707,0.007420,True
83,600016,民生银行,2020,69502.33,1667.77,351.02,0.9221,0.210473,0.005050,True


从公司信息表中选出需要合并的列。

In [12]:
company_small = company[["证券代码", "行业名称", "省份", "城市", "上市日期"]]
company_small.head()

,证券代码,行业名称,省份,城市,上市日期
0,000001,货币金融服务,广东省,深圳市,1991-04-03
1,000002,房地产业,广东省,深圳市,1991-01-29
2,000004,软件和信息技术服务业,广东省,深圳市,1991-01-14
3,000006,房地产业,广东省,深圳市,1992-04-27
4,000007,房地产业,广东省,深圳市,1992-04-13


把 2020 年排名表和公司信息表合并。

In [13]:
rank_with_info = finance_2020_rank_with_code.merge(
    company_small,
    on="证券代码",
    how="left",
)

rank_with_info[[
    "证券代码", "证券简称", "行业名称", "省份", "营业收入_亿元", "净利润_亿元", "净利率"
]].head(10)

,证券代码,证券简称,行业名称,省份,营业收入_亿元,净利润_亿元,净利率
0,000002,万科A,房地产业,广东省,4191.12,592.98,0.141485
1,000333,美的集团,电气机械及器材制造业,广东省,2857.10,275.07,0.096276
2,000338,潍柴动力,汽车制造业,山东省,1974.91,112.75,0.057091
3,600000,浦发银行,货币金融服务,上海市,1746.87,589.93,0.337707
4,600016,民生银行,货币金融服务,北京市,1667.77,351.02,0.210473
5,000001,平安银行,货币金融服务,广东省,1432.42,289.28,0.201952
6,600015,华夏银行,货币金融服务,北京市,927.17,215.68,0.232622
7,000016,深康佳A,计算机、通信和其他电子设备制造业,广东省,503.52,5.40,0.010724
8,000550,江铃汽车,汽车制造业,江西省,330.96,5.51,0.016649
9,000538,云南白药,医药制造业,云南省,327.43,55.11,0.168311


合并后可以提出更具体的问题。例如：2020 年营业收入前十的公司分别来自哪些行业？

In [14]:
rank_with_info.head(10)["行业名称"].value_counts()

行业名称
货币金融服务              4
汽车制造业               2
房地产业                1
电气机械及器材制造业          1
计算机、通信和其他电子设备制造业    1
医药制造业               1
Name: count, dtype: int64

阶段 2 小结：这一阶段带出证券代码的字符串处理、补齐 6 位、第二张表、选列、`merge`，以及合并后的分类统计。

## 阶段 3：按行业汇总，重建一张分析表

现在不再只看公司名单，而是比较行业。目标是得到一张行业层面的汇总表。

In [15]:
analysis_df = finance_with_code.merge(company_small, on="证券代码", how="left")
analysis_df.head()

,证券代码,证券简称,年份,总资产_亿元,营业收入_亿元,净利润_亿元,资产负债率,净利率,资产收益率,是否盈利,行业名称,省份,城市,上市日期
0,000001,平安银行,2018,34185.92,1062.12,248.18,0.9298,0.233665,0.007260,True,货币金融服务,广东省,深圳市,1991-04-03
1,000001,平安银行,2019,39390.70,1268.14,281.95,0.9205,0.222333,0.007158,True,货币金融服务,广东省,深圳市,1991-04-03
2,000001,平安银行,2020,44685.14,1432.42,289.28,0.9185,0.201952,0.006474,True,货币金融服务,广东省,深圳市,1991-04-03
3,000002,万科A,2018,15285.79,2976.79,492.72,0.8459,0.165521,0.032234,True,房地产业,广东省,深圳市,1991-01-29
4,000002,万科A,2019,17299.29,3678.94,551.32,0.8436,0.149858,0.031870,True,房地产业,广东省,深圳市,1991-01-29


按行业汇总 2020 年的数据。

In [16]:
industry_2020 = (
    analysis_df[analysis_df["年份"] == 2020]
    .groupby("行业名称")
    .agg(
        公司数=("证券简称", "nunique"),
        平均营业收入_亿元=("营业收入_亿元", "mean"),
        平均净利率=("净利率", "mean"),
        平均资产负债率=("资产负债率", "mean"),
    )
    .sort_values("平均营业收入_亿元", ascending=False)
)

industry_2020

,公司数,平均营业收入_亿元,平均净利率,平均资产负债率
行业名称,,,,
货币金融服务,4,1443.5575,0.245688,0.919075
房地产业,4,1065.4900,-0.526516,0.702300
电气机械及器材制造业,4,813.0450,0.035393,0.676075
汽车制造业,4,631.4550,0.049062,0.542475
计算机、通信和其他电子设备制造业,4,170.3000,0.026226,0.523675
医药制造业,4,124.9825,0.103404,0.338025
软件和信息技术服务业,4,52.4050,0.023724,0.464825


如果希望看到每个行业收入最高的公司，可以用分组循环。

In [17]:
top_list = []

for industry, group in analysis_df[analysis_df["年份"] == 2020].groupby("行业名称"):
    top_company = group.sort_values("营业收入_亿元", ascending=False).head(1)
    top_list.append(top_company)

industry_top_company = pd.concat(top_list)[
    ["行业名称", "证券代码", "证券简称", "营业收入_亿元", "净利率"]
].sort_values("营业收入_亿元", ascending=False)

industry_top_company

,行业名称,证券代码,证券简称,营业收入_亿元,净利率
5,房地产业,000002,万科A,4191.12,0.141485
47,电气机械及器材制造业,000333,美的集团,2857.10,0.096276
50,汽车制造业,000338,潍柴动力,1974.91,0.057091
77,货币金融服务,600000,浦发银行,1746.87,0.337707
20,计算机、通信和其他电子设备制造业,000016,深康佳A,503.52,0.010724
62,医药制造业,000538,云南白药,327.43,0.168311
68,软件和信息技术服务业,000555,神州信息,106.86,0.043608


最后，把三年营业收入重建成一张公司层面的宽表，便于比较收入变化。

In [18]:
revenue_wide = analysis_df.pivot_table(
    index=["证券代码", "证券简称", "行业名称"],
    columns="年份",
    values="营业收入_亿元",
)

revenue_wide["收入增长率_2018_2020"] = revenue_wide[2020] / revenue_wide[2018] - 1

company_summary = revenue_wide.reset_index().sort_values("收入增长率_2018_2020", ascending=False)
company_summary.head(10)

年份,证券代码,证券简称,行业名称,2018,2019,2020,收入增长率_2018_2020
10,000045,深纺织A,计算机、通信和其他电子设备制造业,12.72,21.58,21.09,0.658019
5,000011,深物业A,房地产业,27.87,39.62,41.04,0.472551
9,000030,富奥股份,汽车制造业,78.53,100.64,111.13,0.415128
1,000002,万科A,房地产业,2976.79,3678.94,4191.12,0.407933
0,000001,平安银行,货币金融服务,1062.12,1268.14,1432.42,0.348642
26,600015,华夏银行,货币金融服务,694.03,827.69,927.17,0.335922
27,600016,民生银行,货币金融服务,1288.93,1549.82,1667.77,0.293918
16,000338,潍柴动力,汽车制造业,1592.56,1743.61,1974.91,0.240085
20,000538,云南白药,医药制造业,267.08,296.65,327.43,0.225962
18,000513,丽珠集团,医药制造业,88.61,93.85,105.20,0.187225


保存阶段成果。

In [19]:
finance_2020_rank_with_code.to_excel("data/finance_2020_rank.xlsx", index=False)
industry_2020.to_excel("data/industry_2020_clean_summary.xlsx")
company_summary.to_excel("data/company_clean_summary.xlsx", index=False)

print("已保存三个阶段成果")

已保存三个阶段成果


阶段 3 小结：这一阶段带出 `groupby().agg()`、分组循环、`concat`、`pivot_table` 和结果保存。